1. Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

2. Separating Target Variables (for Survival)

In [2]:
df = pd.read_csv(
    "../data/raw/METABRIC_RNA_Mutation.csv",
    low_memory=False
)

survival_time = df["overall_survival_months"]
survival_event = df["overall_survival"]

#print(survival_time.head())
#print(survival_event.head())

3. Remove Target Columns

In [3]:
X = df.drop(
    columns=[
        "overall_survival_months",
        "overall_survival"
    ]
)

X = X.drop(columns=["patient_id"])
# Note - here, x=predictors; y=survival;
#print(X.shape)

4. Identify Column Types

In [ ]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

numeric_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print(f"Categorical columns: {len(categorical_cols)}") #size of categorical columns
print(f"Numeric columns: {len(numeric_cols)}") #size of numeric columns

5. Missing Value Summary

In [ ]:
missing = (
    X.isnull().mean() * 100
).sort_values(ascending=False)

missing[missing > 0]

Summary Results

|Column                         |Percentage   |
|-------------------------------|-------------|
|tumor_stage                    |   26.313025 |
|3-gene_classifier_subtype      |   10.714286 |
|primary_tumor_laterality       |    5.567227 |
|neoplasm_histologic_grade      |    3.781513 | 
|cellularity                    |    2.836134 |
|mutation_count                 |    2.363445 |
|er_status_measured_by_ihc      |    1.575630 |
|type_of_breast_surgery         |    1.155462 |
|tumor_size                     |    1.050420 |
|tumor_other_histologic_subtype |    0.787815 | 
|oncotree_code                  |    0.787815 |
|cancer_type_detailed           |    0.787815 |
|death_from_cancer              |    0.052521 |


6. Best Imputation Strategy

In [ ]:
# checking the distribution of categorical variables to infer the best imputation strategy for missing values
print("Tumor Stage")
print(df["tumor_stage"].value_counts(dropna=False))
print()

print("Histologic Grade")
print(df["neoplasm_histologic_grade"].value_counts(dropna=False))
print()

print("Cellularity")
print(df["cellularity"].value_counts(dropna=False))
print()

print("Type of Breast Surgery")
print(df["type_of_breast_surgery"].value_counts(dropna=False))

7. Handling Missing Values

In [ ]:
from sklearn.impute import SimpleImputer


# Numeric Features
numeric_imputer = SimpleImputer(strategy="median")

X[numeric_cols] = numeric_imputer.fit_transform(
    X[numeric_cols]
)


# Categorical Features  
categorical_imputer = SimpleImputer(strategy="most_frequent")

X[categorical_cols] = categorical_imputer.fit_transform(
    X[categorical_cols]
)


# Verifying that there are no remaining missing values
remaining_missing = X.isnull().sum().sum()
print(f"Remaining missing values: {remaining_missing}")

Several clinical variables contained missing observations, with `tumor_stage` exhibiting the highest proportion of missing values (~26%). All remaining variables contained fewer than 11% missing observations, indicating that the dataset was generally well curated.

To preserve the maximum number of patients for downstream survival analysis, missing values were imputed rather than removing incomplete records. This approach minimizes the loss of valuable patient information while maintaining statistical power.

**Numerical Features**

Missing values in numerical variables were imputed using the **median**. The median is robust to outliers and preserves the central tendency of the data without being disproportionately influenced by extreme observations, making it particularly suitable for clinical datasets.

**Categorical Features**

Missing values in most categorical variables will be imputed using the **most frequent category (here, the mode)**. Given the relatively low proportion of missing values in these variables, mode imputation provides a simple + widely accepted strategy while preserving the original distribution of categorical labels.

**Tumor Stage**

Although `tumor_stage` contained a comparatively high proportion of missing values, it was retained because tumor stage is a well-established prognostic factor in breast cancer. Median imputation (corresponding to Stage II) was adopted for the baseline model to maximize sample size while maintaining a consistent preprocessing pipeline.

The contribution of `tumor_stage` will be evaluated during model development through feature importance analysis and performance comparisons with alternative feature sets to determine whether it provides additional prognostic value beyond other clinical variables such as tumor size, lymph node involvement, and Nottingham Prognostic Index.________________________________________

8. Encode Categorical Variables

In [ ]:
# Separate mutation columns from other categorical variables
# Mutation columns (all end with '_mut')
mutation_cols = [
    col for col in df.columns
    if col.endswith("_mut")
]

# Remaining clinical categorical columns
clinical_categorical_cols = [
    col for col in categorical_cols
    if col not in mutation_cols
]

print(f"Mutation columns: {len(mutation_cols)}")
print(f"Clinical categorical columns: {len(clinical_categorical_cols)}")

# Binary Encoding for mutation cols
X[mutation_cols] = (
    X[mutation_cols] != "0"
).astype(int)

X[["tp53_mut", "pik3ca_mut"]].head()

X = pd.get_dummies(
    X,
    columns=clinical_categorical_cols,
    drop_first=True,
    dtype=int
)

print(X.shape)

The dataset contained two distinct types of categorical variables requiring different preprocessing strategies:

**Clinical Categorical Variables**

Clinical variables, including surgery type, receptor status, molecular subtype, and other patient characteristics, were transformed using **one-hot encoding**. This representation preserves categorical information without introducing artificial ordinal relationships between categories.

To reduce redundancy and avoid perfect multicollinearity, the first category of each variable was dropped (`drop_first=True`).

**Mutation Features**

Mutation columns differed substantially from standard categorical variables, as they contained detailed amino acid substitutions (e.g., `R175H`, `H1047R`) rather than simple class labels.

Instead of one-hot encoding every mutation subtype—which would have produced thousands of sparse features—each mutation column was converted into a binary indicator:

- **0** = No mutation detected (wildtype)
- **1** = Mutation present

This encoding preserves the clinically relevant information regarding mutation status while substantially reducing dimensionality and mitigating the risk of overfitting.

**Results**
Mutation columns: 173
Clinical categorical columns: 17
(1904, 724)

9. Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler

# define binary cols
binary_cols = [
    col for col in X.columns
    if set(X[col].dropna().unique()).issubset({0, 1})
]

# everything else is continuous
continuous_cols = [
    col for col in X.columns
    if col not in binary_cols
]

print(f"Binary features: {len(binary_cols)}")
print(f"Continuous features: {len(continuous_cols)}")

scaler = StandardScaler()

X[continuous_cols] = scaler.fit_transform(
    X[continuous_cols]
)

# X[continuous_cols].describe().T.head() to verify that means are close to 0 and stdev is close to 1

10. Save Preprocessed Data

In [ ]:
X["overall_survival_months"] = survival_time
X["overall_survival"] = survival_event

X.to_csv(
    "../data/processed/metabric_processed.csv",
    index=False
)